# Session 5: Data Quality & Cleansing Lab

**Course:** Data Analytics Foundation (CompTIA Data+) | **CompTIA Domain:** 5.0 Data Governance, Quality & Controls | 2.0 Data Mining

> Open this notebook in **Google Colab**. Tonight you will play data-quality detective: diagnose a messy customer table, fix it step by step with pandas, and then clean an e-commerce order file on your own. These are the same skills you will apply to your Capstone dataset.

## 🗺️ Your 45-Minute Lab Map
| Part | What you will do | Approx. time |
|---|---|---|
| Part 1 | The 6 dimensions of data quality + pandas cleaning toolkit | 4 min |
| Part 2 | Create the dirty customer DataFrame | 3 min |
| Part 3 | ✏️ Five guided cleaning exercises (nulls, duplicates, text, imputation, outliers) | 15 min |
| Part 4 | Guided Example: fix messy dates and inconsistent categories, then score data quality | 11 min |
| Part 5 | 🧩 Guided Challenge: clean the e-commerce orders file | 12 min |

**How to use this notebook:** read each text cell, then run the code cell beneath it with **Shift + Enter**.
Cells marked **✏️ Your Turn** have blanks (`___`) for you to fill in. Each one has a hidden solution
you can reveal *after* you have tried it. Errors are normal and part of learning. Read the last line
of the error message, fix it, and run the cell again.

## Part 1: Why Data Quality Matters

**Garbage In, Garbage Out (GIGO):** a beautiful dashboard built on dirty data leads to confident but wrong decisions.

### The 6 dimensions of data quality
| Dimension | The question it asks | Example violation |
|---|---|---|
| **Accuracy** | Does the data reflect reality? | A customer's age is recorded as 230 |
| **Completeness** | Is required data present? | Missing annual spend |
| **Consistency** | Do values agree across rows/systems? | `CA`, `Calif`, `California` in one column |
| **Timeliness** | Is it current and available when needed? | An order dated 2031 |
| **Uniqueness** | Is each real-world thing recorded once? | The same customer entered twice |
| **Validity** | Does it follow business rules and formats? | A negative order quantity, or `"not available"` in a date column |

### Your pandas cleaning toolkit
| Problem | Diagnose with | Fix with |
|---|---|---|
| Missing values | `df.isnull().sum()` | `.fillna()` (impute) or `.dropna()` (remove) |
| Duplicates | `df.duplicated().sum()` | `.drop_duplicates()` |
| Messy text | `.value_counts()` | `.str.strip()`, `.str.upper()`, `.str.title()`, `.replace()` |
| Wrong types | `df.dtypes` | `pd.to_datetime()`, `.astype()` |
| Outliers / invalid values | `.describe()`, boolean filters | Boolean filtering, e.g. `df[df["age"].between(18, 100)]` |

## Part 2: Meet Our Dirty Customer Data

Run the cell to create the table. Before reading on, **look closely**. How many problems can you spot? (There are at least six.)

In [ ]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id":  [101, 102, 103, 104, 105, 103, 106, 107, 108, 109, 110, 111],
    "name":         ["  alice johnson", "BOB SMITH ", "Carla Diaz", "dan wu", "Erin Moss", "Carla Diaz",
                     "  Eve Adams", "frank ORTIZ ", "Grace Lee", "hank patel ", "Ivy Chen", "  jake brown"],
    "age":          [34, 45, 29, np.nan, 52, 29, 230, 41, -5, 38, np.nan, 27],
    "state":        ["NY", "ny ", "California", "CA", "Calif", "California", "NY", "new york", "TX", "Texas ", "CA", "tx"],
    "signup_date":  ["2024-01-15", "01/22/2024", "2024-02-03", "March 5, 2024", "2024-03-18", "2024-02-03",
                     "2024/04/01", "2024-04-10", "2024-05-02", "not available", "05-20-2024", "2024-06-01"],
    "annual_spend": [1200.50, 850.00, np.nan, 430.25, 2100.00, np.nan, 990.00, np.nan, 610.00, 1500.00, 720.00, 380.00],
})

raw = customers.copy()     # keep an untouched copy so we can compare before vs. after
customers

## Part 3: ✏️ Five Guided Cleaning Exercises

Work through these in order, because each step builds on the last. Every exercise has a short explanation, a starter cell, and a hidden solution.

### Exercise 1: Find the missing values (Completeness)
`df.isnull()` marks every empty cell as `True`. Adding `.sum()` counts the `True`s in each column.

In [ ]:
# How many missing values are in each column?
customers.___().___()

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
customers.isnull().sum()
```

</details>

**Read the result:** `age` has 2 missing and `annual_spend` has 3 missing. We will decide how to handle them in Exercise 4.

### Exercise 2: Remove duplicate rows (Uniqueness)
Customer 103 (Carla Diaz) appears twice with identical values. First **count** the duplicates, then **remove** them.

In [ ]:
print("Duplicate rows found:", customers.___().sum())

customers = customers.___()
print("Rows remaining:", len(customers))

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
print("Duplicate rows found:", customers.duplicated().sum())
customers = customers.drop_duplicates()
print("Rows remaining:", len(customers))
```

</details>

### Exercise 3: Normalize the text columns (Consistency)
Computers treat `"  alice johnson"` and `"Alice Johnson"` as **different** people. Use these string tools:
- `.str.strip()` removes spaces at the start and end.
- `.str.title()` capitalizes Each Word (good for names).
- `.str.upper()` makes everything UPPERCASE (good for codes like state abbreviations).

You can chain them: `df["col"].str.strip().str.upper()`

In [ ]:
customers["name"]  = customers["name"].str.___().str.___()     # strip, then Title Case
customers["state"] = customers["state"].str.___().str.___()    # strip, then UPPERCASE

customers[["name", "state"]]

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
customers["name"]  = customers["name"].str.strip().str.title()
customers["state"] = customers["state"].str.strip().str.upper()
customers[["name", "state"]]
```

</details>

Better, but `CALIFORNIA`, `CALIF` and `CA` still mean the same state. We fix that in Part 4.

### Exercise 4: Impute missing spend with the median (Completeness)
**Impute** means *fill in a reasonable value*. We use the **median** instead of the mean because the median is not pulled around by extreme values.

In [ ]:
spend_median = customers["annual_spend"].___()
print("Median annual spend:", spend_median)

customers["annual_spend"] = customers["annual_spend"].___(spend_median)
customers["annual_spend"].isnull().sum()     # should now be 0

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
spend_median = customers["annual_spend"].median()
print("Median annual spend:", spend_median)
customers["annual_spend"] = customers["annual_spend"].fillna(spend_median)
customers["annual_spend"].isnull().sum()
```

</details>

### Exercise 5: Filter out invalid ages (Accuracy & Validity)
An age of **230** or **-5** is impossible, so these are data-entry errors. Business rule: customers must be **18 to 100**.

⚠️ **Careful:** two ages are *missing* (NaN), not invalid. We want to keep those rows and impute them. The condition below keeps rows where age is valid **or** (`|`) age is missing.

Afterwards, impute the missing ages with the median of the **valid** ages. *Why must we remove the outliers before computing the median?*

In [ ]:
valid_age = customers["age"].between(___, ___) | customers["age"].isnull()

print("Invalid rows being removed:")
print(customers[~valid_age][["customer_id", "name", "age"]])      # ~ means NOT

customers = customers[valid_age]
customers["age"] = customers["age"].fillna(customers["age"].median())
customers

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
valid_age = customers["age"].between(18, 100) | customers["age"].isnull()
print("Invalid rows being removed:")
print(customers[~valid_age][["customer_id", "name", "age"]])
customers = customers[valid_age]
customers["age"] = customers["age"].fillna(customers["age"].median())
customers
```

</details>

**Answer to the "why":** if 230 were still in the column, it would drag the average and could shift the median, so the "typical" age we impute would be distorted. **Clean first, then compute.**

## Part 4: Guided Example: Messy Dates, Inconsistent Categories, and a Quality Scorecard

### Step 1: Standardize the state categories
`.value_counts()` reveals every spelling. Then a **mapping dictionary** tells pandas "replace this → with that."

In [ ]:
print("BEFORE:")
print(customers["state"].value_counts())

In [ ]:
state_map = {"CALIFORNIA": "CA", "CALIF": "CA", "NEW YORK": "NY", "TEXAS": "TX"}
customers["state"] = customers["state"].replace(state_map)

print("AFTER:")
print(customers["state"].value_counts())

Three clean groups (CA, NY, TX). A `groupby("state")` will now give correct totals instead of splitting California into three buckets.

### Step 2: Parse dates written in many formats
Our `signup_date` column has `2024-01-15`, `01/22/2024`, `March 5, 2024`, `2024/04/01`... and `not available`.

`pd.to_datetime()` with:
- `format="mixed"` means "figure out each value's format individually."
- `errors="coerce"` means "if a value cannot be parsed, put **NaT** (Not a Time) instead of crashing."

In [ ]:
customers["signup_date_clean"] = pd.to_datetime(customers["signup_date"], format="mixed", errors="coerce")
customers[["signup_date", "signup_date_clean"]]

### Step 3: Decide what to do with the unparseable date
Never silently ignore a `NaT`. Find it and make a business decision. Here we **flag** it for follow-up rather than deleting a paying customer.

In [ ]:
customers["date_needs_review"] = customers["signup_date_clean"].isna()
print(customers[customers["date_needs_review"]][["customer_id", "name", "signup_date"]])

# Swap the clean column in for the original
customers = customers.drop(columns="signup_date").rename(columns={"signup_date_clean": "signup_date"})
customers.dtypes

### Step 4: Build a reusable Data Quality Scorecard
Professionals measure quality *before and after* cleaning. This function is reusable, so you can run it on **any** DataFrame, including your Capstone data.

In [ ]:
def quality_report(df, name="data"):
    """Return a one-row summary of common data quality problems."""
    text_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
    untrimmed = sum((df[c].dropna() != df[c].dropna().str.strip()).sum() for c in text_cols)
    return pd.DataFrame({
        "rows": [len(df)],
        "duplicate_rows": [df.duplicated().sum()],
        "missing_cells": [df.isnull().sum().sum()],
        "pct_complete": [round(100 * (1 - df.isnull().sum().sum() / df.size), 1)],
        "untrimmed_text_values": [untrimmed],
    }, index=[name])

pd.concat([quality_report(raw, "BEFORE"), quality_report(customers, "AFTER")])

**💬 Takeaway:** we went from duplicates, untrimmed text and 5 missing cells to a clean, consistent table. The one remaining missing cell is the date we deliberately flagged for review. That scorecard is the kind of evidence you put in your Capstone README.

## Part 5: 🧩 Guided Challenge: Clean the E-Commerce Orders File

The online store exported its orders and the file is a mess. Your job: make it analysis-ready and calculate **total revenue**. Follow the seven steps. Each step names the tool to use.

In [ ]:
orders = pd.DataFrame({
    "order_id":       ["A100", "A101", "A102", "A102 ", "A103", "A104", "A105", "A106", "A107", "A108"],
    "customer_email": ["ann@shop.com", "BOB@SHOP.COM", " cy@shop.com", "cy@shop.com", "dee@shop.com",
                       None, "eli@shop.com", "fay@shop.com", "gus@shop.com", "hal@shop.com"],
    "quantity":       [2, 1, 3, 3, -1, 4, 2, 1, 100, 2],
    "unit_price":     ["$19.99", "$5.00", "$12.50", "$12.50", "$8.75", "$19.99", None, "$5.00", "$12.50", "$1,250.00"],
    "order_date":     ["2025-01-03", "2025-01-04", "2025-01-05", "2025-01-05", "2025-01-06",
                       "2025-01-07", "2025-01-08", "2031-01-09", "2025-01-10", "2025-01-11"],
    "status":         ["shipped", "Shipped ", "SHIPPED", "SHIPPED", "pending", "cancelled", "Shipped", "shipped", "Pending", "shipped"],
})
orders_raw = orders.copy()
quality_report(orders_raw, "orders BEFORE")

**Step 1: Standardize text.** Strip `order_id`; strip and lowercase `customer_email`; strip and Title-case `status`.

In [ ]:
orders["order_id"]       = orders["order_id"].str.___()
orders["customer_email"] = orders["customer_email"].str.___().str.___()
orders["status"]         = orders["status"].str.___().str.___()
orders["status"].value_counts()

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
orders["order_id"]       = orders["order_id"].str.strip()
orders["customer_email"] = orders["customer_email"].str.strip().str.lower()
orders["status"]         = orders["status"].str.strip().str.title()
orders["status"].value_counts()
```

</details>

**Step 2: Remove duplicate orders.** Order `A102` appeared twice. Before Step 1, the trailing space hid it. Each `order_id` should be unique, so dedupe **on that column only** using `subset=`.

In [ ]:
orders = orders.drop_duplicates(subset=___)
print("Rows:", len(orders))

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
orders = orders.drop_duplicates(subset="order_id")
print("Rows:", len(orders))
```

</details>

**Step 3: Fix the price column.** Remove `$` and `,`, convert to float, then impute the one missing price with the median price.

In [ ]:
orders["unit_price"] = (orders["unit_price"]
                        .str.replace("$", "", regex=False)
                        .str.replace(",", "", regex=False)
                        .astype(___))
orders["unit_price"] = orders["unit_price"].fillna(orders["unit_price"].___())
orders["unit_price"]

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
orders["unit_price"] = (orders["unit_price"]
                        .str.replace("$", "", regex=False)
                        .str.replace(",", "", regex=False)
                        .astype(float))
orders["unit_price"] = orders["unit_price"].fillna(orders["unit_price"].median())
orders["unit_price"]
```

</details>

**Step 4: Dates and timeliness.** Convert `order_date` to datetime, then **remove** any order dated in the future (a validity/timeliness error). Use `pd.Timestamp.today()`.

In [ ]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
print("Future-dated orders:\n", orders[orders["order_date"] > pd.Timestamp.today()][["order_id", "order_date"]])

orders = orders[orders["order_date"] ___ pd.Timestamp.today()]

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
orders["order_date"] = pd.to_datetime(orders["order_date"])
print("Future-dated orders:\n", orders[orders["order_date"] > pd.Timestamp.today()][["order_id", "order_date"]])
orders = orders[orders["order_date"] <= pd.Timestamp.today()]
```

</details>

**Step 5: Validity rule.** Quantity must be at least 1. Keep only valid rows.

In [ ]:
orders = orders[orders["quantity"] ___ 1]
print("Rows:", len(orders))

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
orders = orders[orders["quantity"] >= 1]
print("Rows:", len(orders))
```

</details>

**Step 6: Missing email.** Order `A104` has no email. Should we drop the order? *(Revenue is real even if the email is missing!)* Fill it with the placeholder `"unknown"` instead.

In [ ]:
orders["customer_email"] = orders["customer_email"].___("unknown")

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
orders["customer_email"] = orders["customer_email"].fillna("unknown")
```

</details>

**Step 7: Calculate revenue and compare the scorecards.** Create `revenue = quantity × unit_price`. Exclude **cancelled** orders from the revenue total.

In [ ]:
orders["revenue"] = orders["___"] * orders["___"]

completed = orders[orders["status"] != "___"]
print("Total revenue (excluding cancelled): $", round(completed["revenue"].sum(), 2))

pd.concat([quality_report(orders_raw, "BEFORE"), quality_report(orders, "AFTER")])

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
orders["revenue"] = orders["quantity"] * orders["unit_price"]
completed = orders[orders["status"] != "Cancelled"]
print("Total revenue (excluding cancelled): $", round(completed["revenue"].sum(), 2))
pd.concat([quality_report(orders_raw, "BEFORE"), quality_report(orders, "AFTER")])
```

</details>

### ✍️ Reflection (double-click to answer)
1. For each problem you fixed, name the **data quality dimension** it violated.
2. Order **A107** has a quantity of **100** while every other order has 1–4. You did *not* delete it. Should you? What would you do instead?

<details>
<summary><b>💡 Reveal a sample answer</b></summary>

1. Untrimmed/mixed-case text → **Consistency** · duplicate A102 → **Uniqueness** · `$` text prices → **Validity** · missing price/email → **Completeness** · 2031 order → **Timeliness/Validity** · negative quantity → **Validity** (and **Accuracy**).
2. Don't delete it automatically. A quantity of 100 is *suspicious* but could be a real bulk order. **Flag it and confirm with the business** (or the source system). Deleting real data is just as harmful as keeping bad data. In Session 7 you will learn the 1.5 × IQR rule to flag outliers like this objectively.

</details>

---
## ✅ Lab Checklist
- [ ] Diagnosed missing values and duplicates
- [ ] Standardized text and inconsistent categories
- [ ] Imputed missing values with the median (after removing invalid values)
- [ ] Parsed messy date formats and flagged unparseable ones
- [ ] Built a reusable `quality_report()` scorecard
- [ ] Cleaned the e-commerce orders file and calculated revenue

## 💾 Save Your Work to GitHub
1. In Colab, click **File → Save a copy in GitHub**.
2. Choose your course/Capstone repository.
3. Use this commit message: `Session 5: Completed Data Quality & Cleansing Lab`

**Coming up next:** Session 6: Pandas data wrangling: filtering, grouping, merging, and feature engineering. Bring your Capstone dataset!